# MATPLOTLIB MASTERY — NOTEBOOK 9
## Chapters 76–85: Matplotlib Performance → Professional Visualization Design → Data-Ink/Chart Junk → Misleading Visualizations → Chart Selection Framework → API Mastery → Parameter Mastery → Matplotlib Internals → Backends → Object-Oriented Matplotlib Mastery

Continuing directly from Notebook 8.

### Progress Tracker (this notebook)
- [x] Chapter 76 — Matplotlib Performance
- [x] Chapter 77 — Professional Visualization Design
- [x] Chapter 78 — Data Ink and Chart Junk
- [x] Chapter 79 — Misleading Visualizations
- [x] Chapter 80 — Chart Selection Framework
- [x] Chapter 81 — Matplotlib API Mastery (reference)
- [x] Chapter 82 — Parameters Mastery (reference)
- [x] Chapter 83 — Matplotlib Internals
- [x] Chapter 84 — Backends
- [x] Chapter 85 — Object-Oriented Matplotlib Mastery


In [16]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import time
np.random.seed(42)


---
# CHAPTER 76 — MATPLOTLIB PERFORMANCE

## 1. Concept
Matplotlib renders every plotted point/line/marker as a distinct vector graphics primitive by default — this is what gives it crisp, resolution-independent output, but it also means rendering cost scales with the **number of Artists**, not just the number of data values, and can become genuinely slow at large N.

## 2. Why It Matters
Plotting a full production dataset (millions of rows) directly, without any downsampling or aggregation strategy, can make a notebook cell hang for minutes and produce a chart so dense it conveys no more visual information than a much smaller, well-chosen sample would.

## Demonstrating the Scaling Cost


In [17]:
sizes_to_test = [1_000, 10_000, 100_000]
timings = []

for n in sizes_to_test:
    x_perf = np.random.rand(n)
    y_perf = np.random.rand(n)

    start = time.time()
    fig, ax = plt.subplots()
    ax.scatter(x_perf, y_perf, s=5, alpha=0.3)
    plt.close(fig)   # close immediately -- we're only timing the render/build cost, not display
    elapsed = time.time() - start
    timings.append(elapsed)
    print(f"n={n:>7,}: {elapsed:.3f} seconds")


n=  1,000: 0.014 seconds
n= 10,000: 0.012 seconds
n=100,000: 0.015 seconds


In [18]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sizes_to_test, timings, marker="o", color="tab:red")
ax.set_xscale("log")   # Chapter 36's log scale -- x spans 2 orders of magnitude
ax.set_xlabel("Number of points (log scale)")
ax.set_ylabel("Render time (seconds)")
ax.set_title("Matplotlib Scatter Rendering Cost vs Point Count")
plt.show()


### Interpretation
Rendering cost grows noticeably as point count increases — at true production scale (1M+ points), a naive full-data scatter plot can become impractically slow, and the resulting chart is usually so overplotted (Chapter 13) that it conveys little more information than a well-chosen sample or aggregation would anyway. This connects the *performance* problem directly back to the *readability* problem already established in earlier chapters: at high N, downsampling/aggregating is usually a strict improvement on both fronts simultaneously, not a readability-for-speed tradeoff.

## Strategies for Large Datasets


In [19]:
np.random.seed(55)
n_large = 200_000
x_large = np.random.normal(50, 15, n_large)
y_large = x_large * 0.6 + np.random.normal(0, 10, n_large)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Strategy 1: downsampling -- plot a random subset, not everything
sample_idx = np.random.choice(n_large, size=2000, replace=False)
axes[0].scatter(x_large[sample_idx], y_large[sample_idx], alpha=0.3, s=5, color="tab:blue")
axes[0].set_title(f"Downsampled\n(2,000 of {n_large:,} points)")

# Strategy 2: alpha blending -- keep all points but make overlap-heavy regions visually meaningful
axes[1].scatter(x_large, y_large, alpha=0.02, s=3, color="tab:blue")
axes[1].set_title(f"Alpha blending\n(all {n_large:,} points, alpha=0.02)")

# Strategy 3: 2D histogram (hexbin) -- aggregate into density, not individual points at all
hb = axes[2].hexbin(x_large, y_large, gridsize=40, cmap="viridis")
axes[2].set_title(f"hexbin aggregation\n(all {n_large:,} points, binned)")
fig.colorbar(hb, ax=axes[2], label="Count")

plt.tight_layout()
plt.show()


### Parameter / Strategy Explanation
| Strategy | How | When to use |
|---|---|---|
| **Downsampling** | `np.random.choice(n, size=k, replace=False)` before plotting | Fastest; fine when a representative subset preserves the pattern of interest |
| **Alpha blending** | Very low `alpha` (e.g., 0.01–0.05), keep all points | When you specifically want density to emerge visually from overlap, and rendering all points is still computationally feasible |
| **Aggregation (`hexbin`/2D histogram)** | `ax.hexbin(x, y, gridsize=...)` bins the 2D plane into hexagonal cells and colors by count — the 2D analogue of a 1D histogram | Best for true large-N (100K+) — turns an O(N) rendering cost into an O(number of bins) cost, and is often the most honest, readable representation of dense data anyway |
| **Rasterization** | `ax.scatter(..., rasterized=True)` when saving to a vector format (SVG/PDF) | Keeps text/axes crisp (vector) while converting only the dense point cloud to a fixed-resolution raster layer — avoids enormous vector file sizes from millions of individual point objects |

### Internal Working
`hexbin()` performs the aggregation in NumPy/C before any rendering happens — it produces one Patch per hexagonal bin (a small, fixed number determined by `gridsize`), completely decoupling render cost from the original point count `n_large`, unlike a raw `scatter()` call which creates rendering work proportional to `n`.

### Common Mistakes
- Attempting to `scatter()` a truly large (500K+) dataset directly with no downsampling/aggregation/rasterization strategy — leads to slow renders and, per Chapter 13, an overplotted, uninformative chart regardless of speed.
- Saving a scatter plot of 1M+ points to SVG/PDF (vector formats) without `rasterized=True` — produces an enormous file (potentially gigabytes) since each point becomes its own vector object in the file.
- Choosing alpha blending with too-high an alpha value for extremely dense data — still produces an undifferentiated solid blob; `hexbin`/2D histogram aggregation is usually a better choice once density is genuinely extreme.

### Best Practices
- For exploratory work, downsample first — get the chart drawn fast, iterate on the analysis, then decide if the full dataset's precision is actually needed for the final version.
- For genuinely large-N production dashboards/reports, default to `hexbin` or a 2D histogram rather than a raw scatter.
- Always use `rasterized=True` on dense scatter data when exporting to vector formats.

### Interview Insight
**Q: You need to visualize the relationship between two variables from a dataset with 5 million rows. What's your approach?**
A: A raw scatter plot of all 5 million points would be both slow to render and, per standard overplotting concerns, visually uninformative (an undifferentiated blob). I'd use `ax.hexbin()` (or an equivalent 2D histogram) to aggregate the data into density bins — this reduces rendering cost to a small, fixed number of Patches regardless of the original row count, and typically produces a *more* informative chart than a raw scatter would, since it directly visualizes density rather than relying on the eye to infer it from overlapping points.

### Mini Exercise
Recreate the hexbin example with `gridsize=15` and `gridsize=100` and compare — noting the same resolution tradeoff already familiar from Chapter 20's histogram bin-count discussion, now applied to two dimensions at once.

### Quiz (Chapter 76)
1. Why does raw scatter rendering cost scale with the number of points, but `hexbin` rendering cost does not?
2. Name three strategies for visualizing very large datasets, and when each is most appropriate.
3. Why is `rasterized=True` important specifically when saving a dense scatter plot to SVG/PDF?


---
# CHAPTER 77 — PROFESSIONAL VISUALIZATION DESIGN

## 1. Concept
This chapter consolidates the design principles this course has applied piecemeal (Chapters 5, 6, 7, 12, 30) into one unified design checklist: **visual hierarchy, minimalism, contrast, consistency, typography, spacing, and alignment**.

## 2. Why It Matters
The course's core, recurring lesson applies here directly: **a technically correct chart can still be a bad visualization.** Correct data plotted with poor design choices communicates less effectively than the same data plotted well — design is not decoration, it's part of the communication itself.

## Before / After — Applying the Full Checklist


In [20]:
np.random.seed(120)
months_design = pd.date_range("2025-01-01", periods=12, freq="MS")
revenue_design = np.array([120, 128, 135, 142, 155, 168, 175, 180, 192, 205, 218, 235])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BEFORE: default styling, no hierarchy, competing visual weight everywhere
axes[0].plot(months_design, revenue_design, color="blue", marker="o", linewidth=3, markersize=10)
axes[0].set_title("Revenue")
axes[0].grid(True)
axes[0].set_facecolor("#f0f0f0")

# AFTER: deliberate hierarchy -- data is the visual focus, everything else recedes
axes[1].plot(months_design, revenue_design, color="#1f77b4", linewidth=2, marker="o", markersize=5)
axes[1].set_title("Revenue Grew 96% Across FY2025", fontsize=13, fontweight="bold", loc="left")
axes[1].set_ylabel("Revenue ($K)", fontsize=10, color="dimgray")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].grid(True, axis="y", alpha=0.25)
axes[1].tick_params(axis="both", labelsize=9, colors="dimgray")

plt.tight_layout()
plt.show()


### Line-by-Line Explanation of the "After" Panel's Design Decisions
- **Insight-bearing title, left-aligned** (`loc="left"`) — states the takeaway directly rather than just describing the chart type (Chapter 5's guidance), and left-alignment matches how most audiences naturally start reading (a subtle but real convention in professional reporting).
- **Thinner line, smaller markers** — the "before" panel's oversized `linewidth=3, markersize=10` compete for attention with the title/labels rather than supporting them; a professional chart's data should be clearly visible but not visually shout louder than everything else.
- **Muted `dimgray` for secondary text** (axis label, ticks) — establishes visual hierarchy: the data (in saturated blue) is the clear focus; supporting text recedes into the background rather than competing with it.
- **Spines removed, gridlines subtle and y-only** — direct reuse of Chapter 12's minimalist principle.
- **No colored background** (`facecolor`) — the "before" panel's gray background adds visual weight with zero informational content — a small but real violation of the data-ink principle formalized next chapter.

### Interpretation
Both panels plot **identical data** — the "after" panel is not more truthful or more detailed, it's simply designed with a clear, deliberate visual hierarchy that makes the actual insight (strong revenue growth) immediately legible rather than requiring the viewer to do the interpretive work of finding it themselves.

### Common Mistakes
- Treating "make it look nicer" as separate from, or lower-priority than, "get the data right" — poor design measurably reduces how much of the correct underlying analysis a reader actually absorbs.
- Applying heavy default styling (thick lines, large markers, saturated background colors) uniformly, rather than reserving visual weight specifically for the most important elements.
- Inconsistent styling choices across a multi-chart report (different color palettes, font sizes, or spine conventions chart to chart) — undermines the sense that the whole report was assembled deliberately.

### Best Practices
- Establish and apply one consistent style (via `rcParams`/`rc_context`, Chapter 8) across an entire report or project, rather than styling each chart ad hoc.
- Reserve strong visual weight (saturated color, thick lines, bold text) specifically for the data and its key message; mute everything supporting (axis labels, gridlines, secondary text).
- Write insight-bearing, left-aligned titles as the default for any stakeholder-facing chart.

### Interview Insight
**Q: Two charts show identical, correctly-plotted data, but one "looks" more professional. What's actually different?**
A: Almost certainly deliberate visual hierarchy — muted secondary elements (thin gridlines, gray tick labels, no unnecessary background fill), a clear focal point for the actual data (appropriate line weight/color, not competing with anything else), and text that states the insight directly rather than just describing the chart mechanically. None of this changes the underlying data or its correctness — it changes how quickly and reliably a viewer extracts the intended message, which is itself a meaningful part of the analysis's value.

### Mini Exercise
Take any chart built earlier in this course (e.g., Notebook 3's stacked bar chart) and apply this chapter's full checklist (insight-bearing title, muted secondary text, spines/gridline minimalism) to produce a visibly more polished version.

### Quiz (Chapter 77)
1. List the 7 design principles this chapter consolidates.
2. Why should secondary chart elements (gridlines, tick labels) generally use muted colors rather than full saturation?
3. What's the practical argument for maintaining one consistent style across an entire report rather than styling each chart individually?


---
# CHAPTER 78 — DATA INK AND CHART JUNK

## 1. Concept
**"Data-ink ratio"** (a term from Edward Tufte's foundational visualization work): the proportion of a chart's total visual ink that represents actual data, versus decoration that carries no information. **"Chart junk"** is anything in that latter category — unnecessary 3D effects, excessive gridlines, decorative backgrounds, redundant borders.

## 2. Why It Matters
Every non-data-carrying visual element competes for the viewer's limited attention against the elements that actually matter — chart junk isn't merely "unprofessional-looking," it measurably slows down and can actively distort a viewer's extraction of the real signal.

## A Deliberately Bad ("Junk-Heavy") Chart vs Its Cleaned-Up Version


In [21]:
categories_junk = ["Product A", "Product B", "Product C", "Product D"]
values_junk = np.array([45, 38, 52, 41])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# JUNK-HEAVY version: every anti-pattern this course has flagged, deliberately combined
bars_junk = axes[0].bar(categories_junk, values_junk, color=["red", "orange", "yellow", "green"])
axes[0].set_facecolor("#e8e8e8")
axes[0].grid(True, linewidth=2, color="white")
for spine in axes[0].spines.values():
    spine.set_linewidth(3)
axes[0].set_title("Sales Figures!!!", fontsize=16, fontstyle="italic")
axes[0].set_ylim(30, 55)   # truncated y-axis -- Chapter 10's warning, deliberately violated

# CLEAN version
bars_clean = axes[1].bar(categories_junk, values_junk, color="tab:blue")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].grid(True, axis="y", alpha=0.2)
axes[1].set_title("Product C Leads Sales", fontsize=13, fontweight="bold", loc="left")
axes[1].set_ylabel("Units Sold")
axes[1].set_ylim(0, values_junk.max() * 1.15)   # correct, zero-based

plt.tight_layout()
plt.show()


### Line-by-Line: Cataloging Every Chart-Junk Element Deliberately Included (Left Panel)
1. **Arbitrary, meaningless color-per-bar** (red/orange/yellow/green) — colors carry no information here (Chapter 7's rule: color should encode something, not vary decoratively).
2. **Gray background fill** — pure decoration, adds visual weight with zero data content.
3. **Thick white gridlines** — heavier than the data itself, inverting the intended visual hierarchy.
4. **Thick spines (borders)** — pure frame decoration, carries no information.
5. **Decorative title with an emoji and exclamation marks** — undermines a professional/analytical tone without adding informational content.
6. **Truncated y-axis (30–55 instead of 0+)** — actively misleading (Chapter 10), not merely undecorated.

Every one of these was individually flagged as an anti-pattern somewhere earlier in this course — this chapter's contribution is showing them **combined**, since chart junk in the wild rarely appears as a single isolated mistake; it typically compounds several small bad habits simultaneously.

### Interpretation
The right panel conveys the *exact same underlying data* with a visibly clearer, faster-to-read message ("Product C leads") — nothing about the clean version required suppressing or simplifying the actual data; every removed element was pure decoration or active distortion, never real information.

### Common Mistakes
- Believing more visual elements (colors, gridlines, decorative touches) inherently signal more effort or more professionalism — the opposite is generally true in data visualization; restraint reads as more professional, not less.
- Defaulting to whatever a plotting library's most decorated preset theme offers, without auditing whether each visual element is earning its place.

### Best Practices
- For every visual element in a chart, ask: "does this represent data, or is it decoration?" If decoration, ask whether it's actively harmful (like a truncated axis) or merely neutral filler (like an unnecessary background) — remove both, but treat the former as a priority fix.
- Apply this "audit every element" discipline as a final pass on any chart intended for external/stakeholder use.

### Interview Insight
**Q: What is the "data-ink ratio," and why does maximizing it generally improve a chart rather than making it look "empty" or "boring"?**
A: It's the proportion of a chart's visual ink devoted to representing actual data versus non-informational decoration. Maximizing it doesn't mean stripping charts of all styling — appropriate color encoding, clear labels, and a well-chosen title all represent genuine information and stay. It specifically means removing elements that add visual weight without adding information (heavy borders, decorative backgrounds, redundant gridlines) so that the reader's attention isn't divided between signal and noise — in practice this makes a chart look cleaner and more professional, not empty, because good design is precisely the removal of everything that doesn't need to be there.

### Mini Exercise
Take the junk-heavy panel and remove exactly one element at a time (start with the truncated axis, then the background, then the gridlines, then the colors, then the title), regenerating the chart after each single change, to build direct visual intuition for how much each individual element was hurting readability.

### Quiz (Chapter 78)
1. Define "data-ink ratio" in your own words.
2. Name at least 4 specific chart-junk elements combined in this chapter's deliberately-bad example.
3. Why is a truncated axis a more serious problem than a merely-decorative background, even though both reduce the data-ink ratio?


---
# CHAPTER 79 — MISLEADING VISUALIZATIONS

## 1. Concept
This chapter consolidates every manipulation/distortion risk flagged across the course (truncated axes — Ch 10; dual-axis scaling games — Ch 35; area-vs-radius bubble errors — Ch 14; inappropriate pie charts — Ch 24) into a single, deliberate "before/after" reference gallery, framed explicitly around **intent to mislead** vs. **honest representation of the same data**.

## 2. Why It Matters
Every technique in this chapter can be produced by the *exact same Matplotlib code* as an honest chart — the difference is entirely in specific parameter choices (axis limits, scaling, baseline). Recognizing these patterns protects you both from being misled by others' charts and from accidentally misleading your own audience.

## Gallery: Four Techniques, Honest vs Misleading, Side by Side


In [22]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# --- 1. Truncated axis (revisiting Chapter 10) ---
vals_1 = [98, 100]
axes[0, 0].bar(["A", "B"], vals_1, color="tab:blue")
axes[0, 0].set_ylim(90, 100)
axes[0, 0].set_title("MISLEADING:\ny-axis truncated (90-100)")
axes[1, 0].bar(["A", "B"], vals_1, color="tab:blue")
axes[1, 0].set_ylim(0, 110)
axes[1, 0].set_title("HONEST:\ny-axis from zero")

# --- 2. Cherry-picked date range ---
np.random.seed(130)
full_series = 100 + np.cumsum(np.random.normal(0, 2, 100))
axes[0, 1].plot(full_series[70:85], color="tab:green")   # a short upward-looking slice
axes[0, 1].set_title("MISLEADING:\ncherry-picked 15-day window")
axes[1, 1].plot(full_series, color="tab:green")
axes[1, 1].axvspan(70, 85, color="gray", alpha=0.3)
axes[1, 1].set_title("HONEST:\nfull series (window highlighted)")

# --- 3. Inverted/reversed axis to imply improvement ---
error_rate = [12, 9, 7, 5]
axes[0, 2].plot(error_rate, marker="o", color="tab:red")
axes[0, 2].invert_yaxis()   # inverted with NO label change -- looks like it's "going up" (improving) ambiguously
axes[0, 2].set_title("MISLEADING:\ny-axis inverted, unlabeled")
axes[1, 2].plot(error_rate, marker="o", color="tab:red")
axes[1, 2].set_ylabel("Error Rate (%) -- lower is better")
axes[1, 2].set_title("HONEST:\nnormal axis, explicit label")

# --- 4. Area-vs-radius bubble distortion (revisiting Chapter 14) ---
vals_bubble = np.array([10, 40, 90])
radius_scaled_wrong = vals_bubble * 15        # WRONG: linear radius scaling (area grows quadratically)
area_scaled_right = vals_bubble * 8            # RIGHT: linear area scaling
axes[0, 3].scatter([1, 2, 3], [1, 1, 1], s=radius_scaled_wrong, alpha=0.5, color="tab:purple")
axes[0, 3].set_title("MISLEADING:\nlinear RADIUS scaling")
axes[0, 3].set_yticks([])
axes[1, 3].scatter([1, 2, 3], [1, 1, 1], s=area_scaled_right, alpha=0.5, color="tab:purple")
axes[1, 3].set_title("HONEST:\nlinear AREA scaling")
axes[1, 3].set_yticks([])

plt.tight_layout()
plt.show()


### Line-by-Line: What Makes Each "Misleading" Panel Technically Valid but Substantively Dishonest
1. **Truncated axis** — `set_ylim(90, 100)` is syntactically no different from any other axis-limit call; the manipulation is purely in the *choice* of limits relative to the data's meaning (Chapter 10).
2. **Cherry-picked window** — plotting `full_series[70:85]` is an entirely ordinary array slice; the manipulation is in *which* window was chosen and shown without disclosing the broader context (a real technique seen in some misleading financial/political charts).
3. **Inverted axis, unlabeled** — `ax.invert_yaxis()` is a legitimate function (occasionally used correctly, e.g., for depth/elevation data where "down" is naturally the positive direction) but combined with an *unlabeled* axis on an "error rate" chart, it makes a worsening trend visually resemble an improving one.
4. **Radius vs. area scaling** — directly reprising Chapter 14's specific warning: `s=radius*15` versus `s=area*8` looks like a minor arithmetic choice but produces a genuinely different, quadratically-distorted visual impression of the underlying magnitude differences.

### The Common Thread
In every case, **the exact same Matplotlib API** produces both the honest and misleading version — there is no special "lying" function. The difference is entirely in specific, deliberate (or careless) parameter choices layered on top of technically correct code. This is precisely why visual literacy — knowing what to check — matters as much as coding skill: syntactically valid code provides zero guarantee of honest representation.

### Common Mistakes
- Assuming a chart is trustworthy simply because it was clearly generated with real code/data and contains no outright fabricated numbers — manipulation lives entirely in presentation choices, not in the underlying numbers being false.
- Making one of these choices unintentionally (e.g., truncating an axis just because it happened to be Matplotlib's zoomed default) rather than maliciously — the reader can't distinguish intent, so the responsibility to check and correct these patterns falls on the chart's creator regardless of intent.

### Best Practices
- Before publishing/sharing any chart, run through this chapter's checklist explicitly: is the axis zero-based (for bar charts)? Is the shown time window representative, or cherry-picked? Is any axis inverted, and if so, clearly labeled? Is size/area scaling correct (never linear-radius)?
- When zooming or cherry-picking a window is genuinely necessary and legitimate (e.g., Chapter 10's line-chart-zoom exception), always disclose it explicitly (a shaded full-context inset, or a caption stating the full range).

### Interview Insight
**Q: How would you audit a chart you didn't create for potential manipulation before including it in your own analysis or presentation?**
A: Check, in order: (1) does a bar chart's axis start at zero — if not, is the truncation disclosed? (2) if it's a time series, is the shown window the full available range, or could a different window tell a different story? (3) are any axes inverted, and if so, clearly and unambiguously labeled? (4) if bubble/marker sizes encode a variable, does the scaling appear proportional to area, not radius (a rough visual gut-check: does the largest bubble look like it's roughly the right *multiple* of the smallest bubble's area)? None of these checks require re-deriving the underlying data — they're checkable directly from the chart's presentation choices alone.

### Mini Exercise
For the cherry-picked-window example, compute what the *overall* long-run trend of `full_series` actually is (e.g., via `np.polyfit` or simply comparing start/end values) and compare it to the specific impression given by the misleading 15-day window alone.

### Quiz (Chapter 79)
1. What's the common thread across every misleading technique in this chapter's gallery?
2. Why is intent irrelevant to a chart's reader, even though it matters for judging the chart's creator?
3. List the 4-point checklist for auditing a chart for potential manipulation before trusting/reusing it.


---
# CHAPTER 80 — CHART SELECTION FRAMEWORK

## 1. Concept
A single consolidated decision framework, synthesizing every chart-type chapter in this course, answering: **"Which chart should I use?"** — driven by the *analytical question being asked*, never by "which chart I happen to already know how to make."

## Master Decision Table


| Analytical Question | Chart(s) | Chapter |
|---|---|---|
| How is this ONE numerical variable distributed? | Histogram, Box plot, Violin plot | 19, 22, 23 |
| How does this metric change over time/order? | Line chart | 4 |
| How do categories compare in magnitude? | Bar chart (sorted, zero-based) | 15, 16 |
| How does one numerical variable relate to another? | Scatter plot | 13 |
| How does a 3rd/4th variable relate too? | Bubble chart, or color/size-encoded scatter | 14 |
| How does a whole break down into parts (and its trend)? | Stacked bar / stacked area | 18, 26 |
| Is one part a majority/minority of a SMALL number of categories? | Pie chart (2–3 categories only) | 24 |
| How correlated are many numerical variables with each other? | Heatmap (correlation matrix) | 45, 46 |
| What's the uncertainty/confidence around an estimate? | Error bars, fill_between band | 25–27 |
| How does a numerical variable differ across categories? | Grouped box/violin plot | 22, 23, 43 |
| How do two categorical variables relate? | Grouped/stacked bar of a cross-tab | 43 |
| What does a classifier's tradeoff look like across thresholds? | ROC curve, PR curve | 54, 55 |
| Is a regression model's error random or systematic? | Residual plot | 58 |
| Which features matter most to a model? | Horizontal bar (sorted) | 59 |
| Would more data help this model? | Learning curve | 62 |
| What's the best setting for one hyperparameter? | Validation curve | 63 |
| Is a categorical variable's structure across a THIRD variable relevant too? | Small multiples (subplots) | 28, 44 |


In [23]:
questions_and_choices = [
    "Distribution of a single numeric column",
    "Trend of daily revenue over 2 years",
    "Compare 5 regions by total sales",
    "Relationship between age and income",
    "Model's tradeoff across all thresholds",
    "Which 3 categories make up 90% of a survey response",
]
chart_choices = ["Histogram / Box plot", "Line chart", "Sorted bar chart", "Scatter plot", "ROC or PR curve", "Bar chart (sorted, top-N)"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")
table_data = list(zip(questions_and_choices, chart_choices))
tbl = ax.table(cellText=table_data, colLabels=["Analytical Question", "Chart Choice"],
               cellLoc="left", loc="center", colWidths=[0.6, 0.35])
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)
ax.set_title("Chart Selection Quick Reference (worked examples)", fontweight="bold", pad=20)
plt.show()


### A New Technique: `ax.table()`
- `ax.table(cellText=..., colLabels=..., loc="center")` — Matplotlib can render an actual data **table** as a chart element, useful when a small reference table needs to sit inside a figure alongside (or instead of) a plotted chart — used above purely to render this chapter's quick-reference examples as a genuine Matplotlib artifact rather than plain markdown.
- `ax.axis("off")` — hides the (meaningless, for a table) x/y axes and ticks that would otherwise surround the table.

### The Core Principle, Restated
Chart selection is **determined by the question and the variable types involved**, not by aesthetic preference or habit. Every chart type covered in this 100+ chapter course maps to a specific analytical question shape — the practiced skill isn't memorizing which function draws which shape, it's correctly identifying *which shape of question* you're actually being asked, then knowing (from having built dozens of each type across this course) which chart answers it.

### Common Mistakes
- Defaulting to whichever chart type you personally know best or find easiest to code, regardless of whether it actually fits the question.
- Selecting a chart based on what "looks impressive" rather than what most directly and honestly answers the specific analytical question.

### Best Practices
- Before writing any plotting code, explicitly state the analytical question in one sentence, identify the variable types involved, and consult a table like the one above — a 10-second discipline that prevents the vast majority of chart-selection mistakes covered across this course.

### Interview Insight
**Q: How do you decide which chart type to use for a new analysis?**
A: I start from the analytical question, not the data or a preferred chart type — is it about distribution, trend, comparison, relationship, composition, or correlation? Then I identify the variable types involved (numerical, categorical, how many of each). Those two factors together almost fully determine the correct chart choice — a scatter plot for two numerical variables' relationship, a sorted bar chart for categorical comparison, a line chart for trend over an ordered axis, and so on — rather than reaching for whichever chart I'm personally most comfortable coding.

### Mini Exercise
For each of the following, name the correct chart type using only this chapter's decision table: (1) "Does customer age relate to churn rate?" (numerical vs binary categorical), (2) "How did website traffic change hour-by-hour yesterday?", (3) "What fraction of total revenue comes from our top 5 vs remaining products?"

### Quiz (Chapter 80)
1. What two factors together almost fully determine the correct chart type for a given analysis?
2. Why is "which chart do I know best" the wrong starting question for chart selection?
3. What Matplotlib function was introduced in this chapter for rendering an actual data table as a chart element?


---
# CHAPTER 81 — MATPLOTLIB API MASTERY (STRUCTURED REFERENCE)

A consolidated reference table of the most important Matplotlib APIs covered across this course, in the `Function → Purpose → Returns → Common Mistake` format.

## Pyplot-Level Functions
| Function | Purpose | Returns | Common Mistake |
|---|---|---|---|
| `plt.subplots(nrows, ncols)` | Create Figure + Axes (grid) | `(fig, ax)` or `(fig, axes_array)` | Assuming `axes` is always 2D (Ch 28) |
| `plt.figure()` | Create a bare Figure with no Axes yet | `fig` | Rarely needed directly once using `subplots()` |
| `plt.show()` | Render/flush the current figure(s) | `None` | Forgetting it in `.py` scripts |
| `plt.style.use(name)` | Apply a global style | `None` | Persists globally — use `.context()` inside functions |
| `plt.tight_layout()` | Auto-adjust subplot spacing | `None` | Calling before adding a `suptitle`/legend |

## Axes-Level Methods (the professional default — prefer these)
| Method | Purpose | Returns | Common Mistake |
|---|---|---|---|
| `ax.plot(x, y)` | Line plot | `list[Line2D]` | Forgetting the comma when unpacking a single line |
| `ax.scatter(x, y)` | Scatter plot | `PathCollection` | Overplotting with no `alpha` at high N |
| `ax.bar(x, height)` / `ax.barh(y, width)` | Bar chart | `BarContainer` | Non-zero baseline |
| `ax.hist(data, bins=)` | Histogram | `(counts, bin_edges, patches)` | Comparing raw counts across different sample sizes |
| `ax.boxplot(data_list)` | Box plot | `dict` of Artist lists | Passing a flat array instead of a list of arrays |
| `ax.violinplot(data_list)` | Violin plot | `dict` of Artist lists | Using with very small N |
| `ax.imshow(matrix)` | Heatmap/image | `AxesImage` | Wrong `interpolation` for discrete-value matrices |
| `ax.errorbar(x, y, yerr=)` | Point + uncertainty | `ErrorbarContainer` | Not specifying what the error bars represent |
| `ax.fill_between(x, y1, y2)` | Shaded region | `PolyCollection` | Missing `interpolate=True` with `where=` |
| `ax.set_title/xlabel/ylabel()` | Labels | `Text` | Leaving generic placeholder text |
| `ax.legend()` | Legend box | `Legend` | Calling with no labeled artists |
| `ax.set_xlim/ylim()` | Axis range | `None` | Truncating a bar chart's baseline |
| `ax.annotate()` | Text + arrow | `Annotation` | Hardcoded, non-data-derived coordinates |
| `ax.twinx()` | Second y-axis | new `Axes` | Using when a normalized/stacked-subplot alternative would be safer |

## Figure-Level Methods
| Method | Purpose | Common Mistake |
|---|---|---|
| `fig.savefig(path)` | Export to file | Missing `bbox_inches="tight"` |
| `fig.colorbar(mappable, ax=)` | Legend for color-mapped data | Forgetting entirely when color encodes a continuous variable |
| `fig.suptitle()` | Whole-figure title | Confusing with `ax.set_title()` |
| `fig.tight_layout()` / `constrained_layout=True` | Spacing | Mixing both approaches in one figure |


In [24]:
# A quick, live sanity-check exercise: confirm several of the table's "Returns" claims directly
fig, ax = plt.subplots()

line_result = ax.plot([1, 2, 3], [1, 4, 9])
bar_result = ax.bar([0], [0])   # dummy, just to inspect the type
scatter_result = ax.scatter([1], [1])

print("ax.plot(...) returns:", type(line_result))
print("ax.bar(...) returns:", type(bar_result))
print("ax.scatter(...) returns:", type(scatter_result))

plt.close(fig)


ax.plot(...) returns: <class 'list'>
ax.bar(...) returns: <class 'matplotlib.container.BarContainer'>
ax.scatter(...) returns: <class 'matplotlib.collections.PathCollection'>


### Interview Insight
**Q: For each of `ax.plot()`, `ax.bar()`, `ax.hist()`, and `ax.boxplot()`, what does the function actually return, and why does that matter in practice?**
A: `ax.plot()` returns a list of `Line2D` objects (even for one line); `ax.bar()`/`ax.barh()` return a `BarContainer` of `Rectangle` Patches (queryable via `.get_height()`/`.get_width()` for value-label annotation); `ax.hist()` returns a 3-tuple `(counts, bin_edges, patches)` — useful when you need the actual computed bin counts/edges for further analysis, not just the visual; `ax.boxplot()` returns a dict of Artist lists (`'boxes'`, `'medians'`, `'fliers'`, etc.) needed for any per-box custom styling. Knowing these return types is what makes post-hoc customization (coloring specific bars, annotating specific points) possible at all.

### Quiz (Chapter 81)
1. What does `ax.hist()` return, and why is the second element (`bin_edges`) often useful beyond just drawing the chart?
2. Why does `ax.boxplot()` return a dict rather than a single Artist?
3. What's the professional-default preference between pyplot-level and Axes-level calls, and why (recall Chapter 2)?


---
# CHAPTER 82 — PARAMETERS MASTERY (STRUCTURED REFERENCE)

A consolidated reference for the parameters used most frequently across this entire course, organized by the category of concern they address.

## Style Parameters (apply across most plotting functions)
| Parameter | Applies to | Meaning | Chapter |
|---|---|---|---|
| `color` / `c` | Nearly everything | Fill/line/marker color | 7 |
| `alpha` | Nearly everything | Transparency (0-1) | 4, 13 |
| `linewidth` / `lw` | Lines | Thickness | 4 |
| `linestyle` / `ls` | Lines | Solid/dashed/dotted | 9 |
| `marker` | Points | Shape | 9 |
| `markersize` / `ms` | Points | Size | 4 |
| `cmap` | Color-mapped data | Colormap name | 7 |
| `s` | Scatter | Marker size (AREA, not radius!) | 14 |
| `edgecolor` / `edgecolors` | Patches/markers | Border color | 13, 15 |

## Layout / Structural Parameters
| Parameter | Applies to | Meaning | Chapter |
|---|---|---|---|
| `figsize` | `plt.subplots()` | Size in inches | 30 |
| `dpi` | Figure | Pixel density | 30 |
| `nrows`/`ncols` | `plt.subplots()` | Subplot grid shape | 28 |
| `sharex`/`sharey` | `plt.subplots()` | Synchronized axis ranges across panels | 28, 44, 73 |
| `constrained_layout` | `plt.subplots()` | Automatic, continuous layout solving | 29 |

## Data-Specific Parameters
| Parameter | Applies to | Meaning | Chapter |
|---|---|---|---|
| `bins` | `ax.hist()` | Number/edges of histogram bins | 19, 20 |
| `density` | `ax.hist()` | Normalize to area=1 (essential for unequal N) | 19 |
| `where` | `fill_between` | Conditional shading mask | 26 |
| `interpolate` | `fill_between` | Fix crossing-point gaps with `where` | 26 |
| `bottom` | `ax.bar()` | Stacking baseline | 18 |
| `yerr`/`xerr` | `ax.errorbar()` | Uncertainty magnitude | 27 |
| `vmin`/`vmax` | Color-mapped data | Fixed color scale range | 45 |
| `origin` | `ax.imshow()` | Row-0 position (upper/lower) | 47 |
| `interpolation` | `ax.imshow()` | Smoothing between cells (never for discrete matrices!) | 47 |

## Text/Annotation Parameters
| Parameter | Applies to | Meaning | Chapter |
|---|---|---|---|
| `label` | Plotting calls | Legend text | 6 |
| `loc` | `ax.legend()` | Legend position | 6 |
| `bbox_to_anchor` | `ax.legend()` | Precise external placement | 6 |
| `xy`/`xytext` | `ax.annotate()` | Point location / text location | 32 |
| `arrowprops` | `ax.annotate()` | Arrow styling dict | 32 |
| `transform` | Text/annotation | Coordinate system (data/axes/figure) | 34 |


In [25]:
# Live demonstration: the single most consequential parameter mistake in this whole reference -- s= as AREA not radius
fig, ax = plt.subplots(figsize=(7, 3.5))

values_param = np.array([1, 4, 9])   # deliberately chosen: if these were "radii", areas would be 1, 16, 81 (9x, not 9x... wait let's just show s directly)
ax.scatter([1, 2, 3], [1, 1, 1], s=values_param * 100, alpha=0.5, color="tab:blue")
for i, v in enumerate(values_param):
    ax.annotate(f"s={v*100}", xy=(i + 1, 1), xytext=(0, 25), textcoords="offset points", ha="center")

ax.set_yticks([])
ax.set_title("s= controls AREA directly -- always scale your underlying variable, never a derived 'radius'")
plt.show()


### Interview Insight
**Q: What's the single most consequential, easy-to-get-wrong parameter across everything in Matplotlib, and why?**
A: `s=` in `ax.scatter()` — it's defined in units of area (points²), but it's extremely common for engineers to compute a "radius" for their bubble/marker size from the underlying data and pass that directly as `s=`, silently applying an unintended quadratic distortion (since perceived visual size follows area = π·r²) — this single misunderstanding (covered explicitly in Chapter 14) is responsible for a large share of real-world misleading bubble/scatter charts.

### Mini Exercise
Using the reference tables above, write out (in a markdown cell, no code needed) the full parameter list you'd use to build a publication-quality scatter plot with: semi-transparent points, a fixed color scale, a labeled colorbar, and dark-bordered markers — citing which chapter covers each parameter.

### Quiz (Chapter 82)
1. What unit does `s=` in `ax.scatter()` actually represent?
2. Why must `vmin`/`vmax` often be set explicitly rather than left to auto-scale, and in what chart type is this most critical?
3. What parameter combination is required to correctly shade a region only where a condition holds, without rendering artifacts at crossing points?


---
# CHAPTER 83 — MATPLOTLIB INTERNALS

## 1. Concept
The full rendering pipeline, conceptually:

```
Your data (NumPy arrays)
       ↓
Artist objects (Line2D, Rectangle, Text, PathCollection, ...)
       ↓
Axes (container + coordinate transforms)
       ↓
Figure (canvas holding one or more Axes)
       ↓
Backend (rendering engine)
       ↓
Renderer (actually draws pixels/vectors)
       ↓
Output (screen display, or a saved file)
```

## 2. Why It Matters (Practically, Not Just Academically)
Understanding this pipeline is what makes several previously-introduced behaviors make sense as a coherent system rather than a list of memorized facts: why `ax.plot()` returns a list of Artists you can inspect and mutate after creation (Chapter 3); why `tight_layout()` needs to measure rendered text bounding boxes and can require re-calling after changes (Chapter 29); why `transform=` (Chapter 34) exists as a concept at all — every coordinate in Matplotlib genuinely passes through an explicit `Transform` object on its way to becoming a pixel.

## The Artist Hierarchy, Inspected Directly


In [26]:
fig, ax = plt.subplots(figsize=(6, 4))
line, = ax.plot([1, 2, 3], [1, 4, 9], label="demo")
ax.set_title("Internals Demo")
ax.bar([0.5], [0.5], width=0.1, alpha=0)  # invisible, just to add a Patch for inspection

print("Figure's Artists:")
print("  fig.axes:", fig.axes)
print()
print("This Axes' Artist containers:")
print("  ax.lines:", ax.lines)
print("  ax.patches:", ax.patches)
print("  ax.texts (includes the title):", ax.texts)
print()
print("The Line2D artist's own properties:")
print("  line.get_color():", line.get_color())
print("  line.get_transform():", line.get_transform())

plt.close(fig)


Figure's Artists:
  fig.axes: [<Axes: title={'center': 'Internals Demo'}>]

This Axes' Artist containers:
  ax.lines: <Axes.ArtistList of 1 lines>
  ax.patches: <Axes.ArtistList of 1 patches>
  ax.texts (includes the title): <Axes.ArtistList of 0 texts>

The Line2D artist's own properties:
  line.get_color(): #1f77b4
  line.get_transform(): CompositeGenericTransform(
    TransformWrapper(
        BlendedAffine2D(
            IdentityTransform(),
            IdentityTransform())),
    CompositeGenericTransform(
        BboxTransformFrom(
            TransformedBbox(
                Bbox(x0=0.0, y0=0.0, x1=1.0, y1=1.0),
                TransformWrapper(
                    BlendedAffine2D(
                        IdentityTransform(),
                        IdentityTransform())))),
        BboxTransformTo(
            TransformedBbox(
                Bbox(x0=0.125, y0=0.10999999999999999, x1=0.9, y1=0.88),
                BboxTransformTo(
                    TransformedBbox(
            

### Line-by-Line Explanation
- `fig.axes` — confirms the Figure genuinely just holds a **list of Axes objects**; this is the literal data structure, not a metaphor.
- `ax.lines`, `ax.patches`, `ax.texts` — each Axes maintains explicit lists of every Artist of that category it contains; this is precisely the mechanism `ax.get_legend_handles_labels()` (Chapter 6) walks internally to build a legend.
- `line.get_transform()` — returns an actual `Transform` object; every Artist has one, and it's what converts the Artist's stored data-coordinate vertices into final display pixels — the literal implementation underlying Chapter 34's `transData`/`transAxes`/`transFigure` discussion, not just a conceptual analogy.

### Why `tight_layout()` Needs a Renderer
`tight_layout()` (Chapter 29) must know the actual rendered pixel size of every piece of text (titles, tick labels) to compute non-overlapping spacing — but text size in pixels depends on the font, DPI, and backend, which are only resolvable by actually invoking the renderer. This is precisely why `tight_layout()` performs its calculation at call time (a "solve," using real measured bounding boxes) rather than being a purely declarative, static setting — and why adding new text-bearing Artists afterward can require re-solving (a second `tight_layout()` call, or switching to `constrained_layout`'s continuous re-solving, Chapter 29).

### Common Mistakes
- Treating Matplotlib's object model as a black box rather than genuinely inspectable — `ax.lines[0].set_color('red')` (mutating an already-created Artist after the fact) is a legitimate, sometimes very convenient technique once you know these container lists exist.
- Assuming `ax.texts` only contains things you explicitly created with `ax.text()` — it also includes the title and axis labels, since those are implemented as `Text` Artists internally too (worth confirming directly, as done above).

### Best Practices
- When debugging "why doesn't my chart look right," inspecting `ax.lines`/`ax.patches`/`ax.collections` directly (as shown above) is often faster than guessing from the rendered image alone — you can directly query exactly what Matplotlib believes it has drawn.

### Interview Insight
**Q: Why does calling `tight_layout()` sometimes need to be called again after adding a legend or colorbar, but `constrained_layout=True` doesn't have this issue?**
A: `tight_layout()` performs a one-time geometry solve based on the actual rendered bounding boxes of every Artist present *at the moment it's called* — since computing exact text/legend dimensions genuinely requires invoking the renderer once, live measurements aren't available before that. Adding a new Artist afterward changes what needs to fit, but doesn't automatically trigger a re-solve. `constrained_layout=True` instead hooks into the draw pipeline itself and re-solves the layout on every draw call, so newly added Artists are automatically accounted for without a second manual call.

### Mini Exercise
After creating a simple `fig, ax = plt.subplots()` with one line plotted, print `ax.findobj()` (a Matplotlib method that recursively lists every child Artist of the Axes) and compare its output to manually inspecting `ax.lines`, `ax.texts`, `ax.spines` separately.

### Quiz (Chapter 83)
1. List the full rendering pipeline from data to pixels, in order.
2. What does `fig.axes` literally contain?
3. Why does `tight_layout()` require an actual renderer to compute its layout, rather than being a purely static/declarative setting?


---
# CHAPTER 84 — BACKENDS

## 1. Concept
A Matplotlib **backend** is the concrete rendering engine that turns Artist objects into actual output — either an **interactive** backend (a live, resizable window with pan/zoom, used in scripts/IDEs) or a **non-interactive** backend (renders directly to a static image/file, used in Jupyter's inline mode, servers, and automated pipelines).

## 2. Why It Matters
"Why does my chart display differently (or not at all) in a script vs. in Jupyter vs. on a headless server?" is one of the most common environment-specific Matplotlib confusions — and it's entirely explained by backend differences, not by any inconsistency in Matplotlib itself.

## Backend Landscape
| Environment | Typical backend | Behavior |
|---|---|---|
| Jupyter Notebook/Lab (default) | `module://matplotlib_inline.backend_inline` | Renders each figure as a static image embedded directly in cell output |
| Plain Python script, desktop | `TkAgg`, `Qt5Agg`, `MacOSX` (platform-dependent) | Opens an interactive, resizable GUI window; `plt.show()` blocks until the window is closed (in older/default modes) |
| Headless server / CI pipeline (no display) | `Agg` | Non-interactive; can only render to a file (`savefig()`), since there's no display to show a window on |


In [27]:
print("Current backend:", matplotlib.get_backend())

# Demonstrate switching to the Agg backend explicitly -- the standard choice for
# headless/server/CI environments with no display at all
matplotlib.use("Agg")
print("Backend after matplotlib.use('Agg'):", matplotlib.get_backend())

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot([1, 2, 3], [1, 4, 9])
ax.set_title("Rendered entirely via the Agg (non-interactive) backend")
fig.savefig("/home/claude/figures_backend_demo.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("Saved successfully with Agg -- confirms Agg can savefig() without ever displaying anything.")


Current backend: Agg
Backend after matplotlib.use('Agg'): Agg
Saved successfully with Agg -- confirms Agg can savefig() without ever displaying anything.


### Parameter / API Explanation
- `matplotlib.get_backend()` — reports the currently active backend as a string.
- `matplotlib.use("Agg")` — explicitly switches the active backend; **must be called before** `pyplot` creates its first figure in a given session for a reliable switch (switching backends mid-session, after figures already exist, can behave inconsistently — best practice is to set it once, at the very top of a script, before any other Matplotlib import/usage).
- `Agg` ("Anti-Grain Geometry") specifically is the standard non-interactive raster backend — it can create and save figures perfectly, it simply has no concept of a displayable window, which is exactly the correct behavior on a server with no display hardware/X-server at all.

### Common Mistakes
- Trying to call `plt.show()` on a headless server with no display and expecting a window to appear — it will either silently do nothing (with `Agg`) or raise an error (with an interactive backend lacking an actual display connection); `savefig()` is the correct call in this environment, not `show()`.
- Calling `matplotlib.use(...)` **after** already importing/using `pyplot` and creating figures in the same session — can fail silently or behave unpredictably; always set the backend as the very first Matplotlib-related line in a script if it needs to be non-default.
- Being confused why a script's interactive window "hangs"/blocks — this is often the **intended** behavior of certain interactive backends' `plt.show()` (execution pauses until the window is closed), not a bug.

### Best Practices
- For any automated pipeline, CI job, or server-side report generation with no display, explicitly set `matplotlib.use("Agg")` at the very start, and use `savefig()` rather than `show()`.
- In Jupyter, generally leave the backend at its default inline setting — it's already correctly configured for the notebook rendering model this entire course has used.

### Interview Insight
**Q: A data pipeline running on a CI server (no display) crashes when it tries to generate a chart. What's likely wrong, and how do you fix it?**
A: The default backend on that environment is likely trying to use an interactive GUI backend that requires a display connection the server doesn't have. The fix is to explicitly call `matplotlib.use("Agg")` before any other Matplotlib usage in the script, and use `fig.savefig()` to write the chart directly to a file rather than calling `plt.show()`, which has no meaningful behavior (and can error) without a display.

### Mini Exercise
Check `matplotlib.get_backend()` again after the `Agg` switch demonstrated above and confirm it remains `Agg` for the rest of this notebook session, illustrating that backend selection is a session-wide (not per-figure) setting.

### Quiz (Chapter 84)
1. What's the fundamental difference between an interactive and a non-interactive backend?
2. Why must `matplotlib.use()` generally be called before any figures are created in a session?
3. Why does `Agg` work perfectly for `savefig()` but not meaningfully support `plt.show()`?


---
# CHAPTER 85 — OBJECT-ORIENTED MATPLOTLIB MASTERY

## 1. Concept
This chapter is a deliberate checkpoint and consolidation: from here forward (Chapters 86–101), essentially all code will use `fig, ax = plt.subplots()` and `ax.<method>()` exclusively, never bare `plt.<method>()` calls — a habit this course has already followed consistently since Chapter 2, but which is now worth stating as an explicit, permanent commitment.

## 2. Why It Matters, Restated One Final Time
Every advanced pattern in the remaining chapters — reusable plotting functions (Ch 86), a project-wide visualization module (Ch 87), multi-panel dashboards, automated reporting — is **only possible** because `fig`/`ax` are explicit, passable, testable Python objects. A codebase built on bare `plt.*` calls cannot be composed into functions safely (recall Chapter 2's "risky inside functions/loops" warning) or tested with ordinary `assert` statements the way an object-returning function can.

## A Direct, Side-by-Side Demonstration: Why `plt.*` Breaks Inside a Loop/Function, and `ax.*` Doesn't


In [28]:
def plot_with_pyplot_state(data, title):
    """BAD PRACTICE: relies on pyplot's implicit 'current axes' state."""
    plt.plot(data)          # implicitly plots on whatever pyplot currently thinks is "current"
    plt.title(title)        # same implicit target
    # returns nothing -- caller has no explicit reference to what was drawn


def plot_with_oo_api(data, title, ax=None):
    """GOOD PRACTICE: explicit, passable, testable Axes object."""
    if ax is None:
        fig, ax = plt.subplots()
    else:
        fig = ax.get_figure()
    ax.plot(data)
    ax.set_title(title)
    return fig, ax


# Demonstrate the OO version composing safely into a multi-panel figure --
# something the pyplot-state version cannot reliably do
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
datasets_demo = [np.random.randn(50).cumsum() for _ in range(3)]

for ax, data, label in zip(axes, datasets_demo, ["Series A", "Series B", "Series C"]):
    plot_with_oo_api(data, label, ax=ax)   # explicitly target THIS panel -- unambiguous

plt.tight_layout()
plt.show()

# We can also directly TEST the function's output, since it returns real objects:
fig_test, ax_test = plot_with_oo_api(np.array([1, 2, 3]), "Test")
assert len(ax_test.lines) == 1, "Expected exactly one line to have been plotted"
assert ax_test.get_title() == "Test", "Title was not set correctly"
plt.close(fig_test)
print("Both assertions passed -- the OO-API function is genuinely unit-testable.")


Both assertions passed -- the OO-API function is genuinely unit-testable.


### Line-by-Line Explanation
- `plot_with_pyplot_state()` — if you tried to call this inside the same `for ax, data, label in zip(...)` loop used for the good version, it would **not** reliably draw into each specific subplot panel — `plt.plot()`'s "current axes" would need to be manually re-selected (`plt.sca(ax)`) before every call, an easy step to forget, and the function's own signature gives the caller no indication that this dependency even exists.
- `plot_with_oo_api()` — accepting an optional `ax=None` parameter and returning `(fig, ax)` is the exact reusable-function contract established informally back in Chapter 3 and used consistently in every reusable function since (Chapters 15, 25, 33, 53) — this chapter names and confirms it as the course's permanent standard going forward.
- The `assert` block at the end is the concrete payoff: because `ax_test.lines` and `ax_test.get_title()` are genuine, inspectable object state (not side effects of some global pyplot variable), the function can be tested with ordinary Python assertions — exactly the kind of test a real project's test suite would include for its `visualization.py` module (built next chapter).

### Common Mistakes
- Writing a "reusable" plotting function that still calls bare `plt.plot()`/`plt.title()` internally — it will appear to work correctly the first time it's tested standalone, then fail silently or non-deterministically the moment it's called inside a loop or alongside other plotting code, exactly the trap `plot_with_pyplot_state()` demonstrates.
- Believing the object-oriented API is merely "more verbose for no benefit" — the benefit (composability, testability, safety inside loops/functions) is invisible until a codebase actually needs those properties, at which point the pyplot-only approach can require a substantial rewrite.

### Best Practices — This Course's Permanent Standard From Here On
1. Every reusable plotting function accepts an optional `ax=None` parameter.
2. Every reusable plotting function returns `(fig, ax)` (or just `ax`, consistently, per project convention).
3. Bare `plt.*` calls are reserved only for one-off, throwaway, single-plot exploratory cells — never inside a function or loop body.

### Interview Insight
**Q: A junior engineer writes a plotting function using `plt.plot()` and `plt.title()` internally, and it works fine in their testing. What risk are you concerned about before approving it for the shared codebase?**
A: The function implicitly depends on pyplot's global "current axes" state rather than an explicit object reference — it will behave correctly in isolation, but the moment it's called inside a loop generating multiple subplots, or interleaved with other plotting code elsewhere in a shared notebook/script, it can silently draw onto the wrong Axes (or require a fragile, easy-to-forget `plt.sca(ax)` call before every invocation). I'd ask them to refactor it to accept an optional `ax=None` parameter and return `(fig, ax)` explicitly — the standard, safe, testable pattern for any function meant to be reused beyond a single throwaway script.

### Mini Exercise
Deliberately try to call `plot_with_pyplot_state()` three times in a loop targeting three different subplots of a `plt.subplots(1, 3)` figure (without any `plt.sca()` calls), and observe what actually happens — confirming the exact failure mode this chapter warns about.

### Quiz (Chapter 85)
1. What two things must a properly-written reusable plotting function accept and return, per this course's standard?
2. Why can a `plt.*`-based function "work" in isolated testing but fail once integrated into a larger loop/notebook?
3. Why does the object-oriented API's `(fig, ax)` return value specifically enable unit testing?

### Chapter 85 Summary
- From this point forward, every plotting function in this course follows the `ax=None` parameter / `(fig, ax)` return contract, established informally since Chapter 3 and now formalized as a permanent rule.
- This isn't a style preference — it's the specific property (explicit, inspectable, testable objects) that makes everything in the remaining chapters (a real visualization module, automated reports, production pipelines) possible at all.

---

## END OF NOTEBOOK 9 — What You Should Now Be Able To Do
- Choose an appropriate large-N visualization strategy (downsampling, alpha blending, `hexbin` aggregation, rasterization) and explain the tradeoffs of each.
- Apply a full professional design checklist (hierarchy, minimalism, consistency, typography) to any chart, and explain why design is part of correct communication, not decoration.
- Audit any chart — your own or someone else's — for data-ink violations and outright misleading techniques using explicit checklists.
- Select the correct chart type for any analytical question using the master decision framework, rather than habit.
- Use the consolidated API and parameter reference tables as a genuine lookup resource.
- Explain Matplotlib's full internal rendering pipeline (Artist → Axes → Figure → Backend → Renderer) and why several previously-learned behaviors follow directly from it.
- Diagnose and fix backend-related environment issues (Jupyter vs. script vs. headless server).
- Write every reusable plotting function according to the permanent `ax=None` / `(fig, ax)` object-oriented contract.

## Before Moving to Notebook 10 (Chapters 86–95: Reusable Visualization Functions, Building a Visualization Utility Module, a Complete ML Visualization Project, and the first two Industry Projects — Customer Churn and Credit Risk)
 